# Clothing Search — Performance Assessment (Config A & B)

Measures search quality for Config A and Config B with a fixed (frozen) CLIP backbone.

---

### Evaluation Metrics
| Measure | Explanation |
|--------|-------------|
| Recall@K | Binary hit — 1 when the correct item appears within the top-K results |
| NDCG@K | Rank-discounted gain — earlier correct hits yield a higher value |
| mAP@K | Averaged precision — rewards retrieval of all relevant items at higher ranks |

K ∈ {5, 10, 15}

---

### Pipeline Overview
1. Embed query images via CLIP (visual features only — captions are not used)
2. Query the FAISS gallery index to obtain top-K candidates
3. Compare candidates against ground truth (matching item_id means correct)
4. Calculate metrics per query and aggregate across random seeds

---

### Required Inputs
- `vr-yolo-bbox-cropped-images` — query image crops and master_crops.csv
- clip indexes dataset — FAISS indexes + item_index_map.csv

## Step 1 — Install Dependencies

In [1]:
!pip uninstall -y faiss faiss-gpu
!pip install ftfy regex transformers faiss-cpu --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 585.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 46.4 MB/s eta 0:00:00


## Step 2 — Import Libraries

In [2]:
import os
import json
import numpy as np
import pandas as pd
import torch
import faiss
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel
import warnings
warnings.filterwarnings('ignore')

GPU = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Runtime device : {GPU}')
print('Imports complete!')

Runtime device : cuda
Imports complete!


## Step 3 — Set Configuration

**Update `ROLL_SEEDS` with your team's roll numbers before running.**

In [3]:
# -----------------------------------------------------------
#  INSERT TEAM ROLL NUMBERS BELOW AS RANDOM SEEDS
# -----------------------------------------------------------
ROLL_SEEDS = [35, 537, 580, 550]   # substitute your actual team roll numbers here
# -----------------------------------------------------------

BBOX_CROPS_DIR  = '/kaggle/input/datasets/akibatra25/vr-yolo-bbox-cropped-images'
INDEXES_DIR     = '/kaggle/input/datasets/likithareddy2508/clip-ab-output-dataset'  # update after saving

K_LIST      = [5, 10, 15]
BATCH_SZ   = 64
CLIP_CKPT   = 'openai/clip-vit-base-patch32'

# Each entry: (label, index file, blend weight)
EVAL_CONFIGS = [
    ('Config_A_beta1.0', 'idx_A_b10.bin', 1.0),
    ('Config_B_beta0.7', 'idx_B_b07.bin', 0.7),
    ('Config_B_beta0.5', 'idx_B_b05.bin', 0.5),
]

for tag, p in [('BBOX_CROPS_DIR', BBOX_CROPS_DIR), ('INDEXES_DIR', INDEXES_DIR)]:
    status_msg = 'Found ✓' if os.path.exists(p) else 'NOT FOUND ✗'
    print(f'[{status_msg}] {tag}')

print(f'\nRoll seeds : {ROLL_SEEDS}')
print(f'K values   : {K_LIST}')

[Found ✓] BBOX_CROPS_DIR
[Found ✓] INDEXES_DIR

Roll seeds : [35, 537, 580, 550]
K values   : [5, 10, 15]


## Step 4 — Load Queries and Gallery Index

In [4]:
master_df = pd.read_csv(os.path.join(BBOX_CROPS_DIR, 'master_crops.csv'))
query_df  = master_df[master_df['split'] == 'query'].reset_index(drop=True)

def translate_path(saved_path):
    if pd.isna(saved_path): return saved_path
    for pfx in ['/kaggle/working/', '/kaggle/input/']:
        if saved_path.startswith(pfx):
            tail = saved_path.replace(pfx, '')
            for ds in ['yolo-bbox-crops-v1/', 'datasets/harshitabansal307/yolo-bbox-crops-v1/']:
                tail = tail.replace(ds, '')
            return os.path.join(BBOX_CROPS_DIR, tail)
    return saved_path

query_df['img_path'] = query_df['crop_path'].apply(translate_path)
query_df['on_disk']  = query_df['img_path'].apply(
    lambda p: os.path.exists(p) if isinstance(p, str) else False
)

if query_df['on_disk'].sum() < len(query_df) * 0.9:
    def direct_path(img_name):
        rel = img_name[4:] if img_name.startswith('img/') else img_name
        for sub in ['data/bbox_crops', 'data/yolo_crops']:
            p = os.path.join(BBOX_CROPS_DIR, sub, rel)
            if os.path.exists(p): return p
        return os.path.join(BBOX_CROPS_DIR, 'data/bbox_crops', rel)
    query_df['img_path'] = query_df['image_name'].apply(direct_path)
    query_df['on_disk']  = query_df['img_path'].apply(os.path.exists)

active_queries = query_df[query_df['on_disk']].reset_index(drop=True)
gallery_lookup  = pd.read_csv(os.path.join(INDEXES_DIR, 'item_index_map.csv'))

print(f'Valid query images  : {len(active_queries):,}')
print(f'Gallery index rows  : {len(gallery_lookup):,}')
print(f'Unique query items  : {active_queries["item_id"].nunique():,}')

Valid query images  : 14,218
Gallery index rows  : 12,612
Unique query items  : 3,985


## Step 5 — Initialise CLIP and Embed Query Images

In [5]:
print(f'Loading CLIP: {CLIP_CKPT}')
clip_processor = CLIPProcessor.from_pretrained(CLIP_CKPT)
clip_model  = CLIPModel.from_pretrained(CLIP_CKPT).to(GPU)
for p in clip_model.parameters():
    p.requires_grad = False
clip_model.eval()
EMBED_DIM = clip_model.config.projection_dim
print(f'CLIP loaded! Vector dim: {EMBED_DIM}')

# Compute query embeddings once; reused across all configs (vision-only)
num_queries     = len(active_queries)
query_embs  = np.zeros((num_queries, EMBED_DIM), dtype=np.float32)

print(f'Encoding {num_queries:,} query images...')
for s in tqdm(range(0, num_queries, BATCH_SZ), desc='Query encoding'):
    chunk   = active_queries.iloc[s : s + BATCH_SZ]
    imgs, ok_idx = [], []
    for i, (_, row) in enumerate(chunk.iterrows()):
        try:
            imgs.append(Image.open(row['img_path']).convert('RGB'))
            ok_idx.append(i)
        except Exception: pass
    if not imgs: continue
    inp = clip_processor(images=imgs, return_tensors='pt', padding=True).to(GPU)
    with torch.no_grad():
        raw  = clip_model.get_image_features(**inp)
        vecs = raw.pooler_output if hasattr(raw, 'pooler_output') and not isinstance(raw, torch.Tensor) else raw
    vecs = vecs / vecs.norm(dim=-1, keepdim=True)
    for li, gi in enumerate(ok_idx):
        query_embs[s + gi] = vecs[li].cpu().numpy()

print(f'Query vectors shape: {query_embs.shape}')

Loading CLIP: openai/clip-vit-base-patch32


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP loaded! Vector dim: 512
Encoding 14,218 query images...


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]


Query encoding: 100%|██████████| 223/223 [02:01<00:00,  1.84it/s]

Query vectors shape: (14218, 512)


## Step 6 — Define Scoring Functions

In [6]:
def compute_recall(top_results, q_id, k):
    return int(any(r == q_id for r in top_results[:k]))


def compute_ndcg(top_results, q_id, n_relevant, k):
    dcg = 0.0
    for rank, rid in enumerate(top_results[:k], start=1):
        if rid == q_id:
            dcg += 1.0 / np.log2(rank + 1)
    ideal_hits = min(n_relevant, k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0


def compute_ap(top_results, q_id, n_relevant, k):
    hits = 0
    running_prec = 0.0
    for rank, rid in enumerate(top_results[:k], start=1):
        if rid == q_id:
            hits += 1
            running_prec += hits / rank
    denom = min(n_relevant, k)
    return running_prec / denom if denom > 0 else 0.0


print('Metric functions ready ✓')
_test_ret = ['A', 'B', 'A', 'C', 'A']
assert compute_recall(_test_ret, 'A', 5) == 1
print(f'Test Recall@5 : {compute_recall(_test_ret, "A", 5)}  ✓')

Metric functions ready ✓
Test Recall@5 : 1  ✓


## Step 7 — Execute Evaluation Loop

In [7]:
gallery_ids    = gallery_lookup['item_id'].tolist()
gallery_counts = gallery_lookup['item_id'].value_counts().to_dict()
eval_records = []

for config_name, index_file, beta in EVAL_CONFIGS:
    print(f'\n=== {config_name} ===')

    index_filepath = os.path.join(INDEXES_DIR, index_file)
    faiss_idx = faiss.read_index(index_filepath)
    print(f'  Index: {faiss_idx.ntotal:,} vectors')

    seed_scores = {k: {'recall': [], 'ndcg_scores': [], 'ap': []} for k in K_LIST}

    for seed in ROLL_SEEDS:
        np.random.seed(seed)
        torch.manual_seed(seed)

        sample_size   = min(max(500, int(0.2 * len(active_queries))), len(active_queries))
        sampled_indices    = np.random.choice(len(active_queries), sample_size, replace=False)
        sampled_embs   = query_embs[sampled_indices].astype(np.float32)
        sampled_ids    = active_queries.iloc[sampled_indices]['item_id'].tolist()

        _, hits  = faiss_idx.search(sampled_embs, 16)

        recall_scores  = {k: [] for k in K_LIST}
        ndcg_scores = {k: [] for k in K_LIST}
        ap_scores  = {k: [] for k in K_LIST}

        for q_id, neighbor_ids in zip(sampled_ids, hits):
            top_results = [gallery_ids[i] for i in neighbor_ids if i < len(gallery_ids)]
            num_relevant     = gallery_counts.get(q_id, 1)
            for k in K_LIST:
                recall_scores[k].append(compute_recall(top_results, q_id, k))
                ndcg_scores[k].append(compute_ndcg(top_results, q_id, num_relevant, k))
                ap_scores[k].append(compute_ap(top_results, q_id, num_relevant, k))

        for k in K_LIST:
            seed_scores[k]['recall'].append(np.mean(recall_scores[k]))
            seed_scores[k]['ndcg_scores'].append(np.mean(ndcg_scores[k]))
            seed_scores[k]['ap'].append(np.mean(ap_scores[k]))

        print(f'  Seed {seed}: R@10={np.mean(recall_scores[10]):.4f}  NDCG@10={np.mean(ndcg_scores[10]):.4f}  mAP@10={np.mean(ap_scores[10]):.4f}')

    for k in K_LIST:
        eval_records.append({
            'config': config_name, 'K': k,
            'Recall_mean': np.mean(seed_scores[k]['recall']),
            'Recall_std' : np.std(seed_scores[k]['recall']),
            'NDCG_mean'  : np.mean(seed_scores[k]['ndcg_scores']),
            'NDCG_std'   : np.std(seed_scores[k]['ndcg_scores']),
            'mAP_mean'   : np.mean(seed_scores[k]['ap']),
            'mAP_std'    : np.std(seed_scores[k]['ap']),
        })

print('\nEvaluation complete!')


=== Config_A_beta1.0 ===
  Index: 12,612 vectors
  Seed 35: R@10=0.5508  NDCG@10=0.2352  mAP@10=0.1644
  Seed 537: R@10=0.5456  NDCG@10=0.2243  mAP@10=0.1534
  Seed 580: R@10=0.5656  NDCG@10=0.2412  mAP@10=0.1684
  Seed 550: R@10=0.5621  NDCG@10=0.2407  mAP@10=0.1690

=== Config_B_beta0.7 ===
  Index: 12,612 vectors
  Seed 35: R@10=0.5610  NDCG@10=0.2356  mAP@10=0.1645
  Seed 537: R@10=0.5424  NDCG@10=0.2235  mAP@10=0.1540
  Seed 580: R@10=0.5614  NDCG@10=0.2373  mAP@10=0.1662
  Seed 550: R@10=0.5565  NDCG@10=0.2385  mAP@10=0.1688

=== Config_B_beta0.5 ===
  Index: 12,612 vectors
  Seed 35: R@10=0.5093  NDCG@10=0.2014  mAP@10=0.1372
  Seed 537: R@10=0.4995  NDCG@10=0.1942  mAP@10=0.1310
  Seed 580: R@10=0.5100  NDCG@10=0.2036  mAP@10=0.1390
  Seed 550: R@10=0.5114  NDCG@10=0.2083  mAP@10=0.1445

Evaluation complete!


## Step 8 — Display and Save Results

In [8]:
metrics_df = pd.DataFrame(eval_records)

print('\n=== ABLATION RESULTS (Config A and B) ===')
print(f'Seeds: {ROLL_SEEDS}  |  Format: mean ± std')
print()

for config_name, _, _ in EVAL_CONFIGS:
    rows = metrics_df[metrics_df['config'] == config_name]
    print(f'Config: {config_name}')
    print(f'  {"K":>4}  {"Recall@K":>14}  {"NDCG@K":>14}  {"mAP@K":>14}')
    print(f'  {"-"*52}')
    for _, r in rows.iterrows():
        print(f'  K={int(r["K"]):>2}  '
              f'{r["Recall_mean"]:.4f}±{r["Recall_std"]:.4f}  '
              f'{r["NDCG_mean"]:.4f}±{r["NDCG_std"]:.4f}  '
              f'{r["mAP_mean"]:.4f}±{r["mAP_std"]:.4f}')
    print()

metrics_df.to_csv('/kaggle/working/eval_results_AB.csv', index=False)
print('Results saved to eval_results_AB.csv')


=== ABLATION RESULTS (Config A and B) ===
Seeds: [35, 537, 580, 550]  |  Format: mean ± std

Config: Config_A_beta1.0
     K        Recall@K          NDCG@K           mAP@K
  ----------------------------------------------------
  K= 5  0.4904±0.0080  0.2327±0.0066  0.1719±0.0059
  K=10  0.5560±0.0081  0.2353±0.0068  0.1638±0.0063
  K=15  0.5916±0.0060  0.2419±0.0066  0.1642±0.0062

Config: Config_B_beta0.7
     K        Recall@K          NDCG@K           mAP@K
  ----------------------------------------------------
  K= 5  0.4762±0.0089  0.2271±0.0058  0.1689±0.0053
  K=10  0.5553±0.0077  0.2337±0.0060  0.1634±0.0056
  K=15  0.5964±0.0054  0.2409±0.0059  0.1641±0.0057

Config: Config_B_beta0.5
     K        Recall@K          NDCG@K           mAP@K
  ----------------------------------------------------
  K= 5  0.4259±0.0048  0.1938±0.0049  0.1415±0.0046
  K=10  0.5076±0.0047  0.2019±0.0051  0.1379±0.0048
  K=15  0.5540±0.0082  0.2101±0.0056  0.1392±0.0051

Results saved to eval_results_